In [13]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [14]:
# Load raw traffic file
traffic_raw = pd.read_csv("cleaned_traffic_data.csv")

# Basic cleaning
traffic_raw["Timestamp"] = pd.to_datetime(traffic_raw["Timestamp"])
traffic_raw["Station"] = traffic_raw["Station"].astype(str)

# Keep only the columns we need for now
traffic_core = traffic_raw[["Timestamp", "Station", "Route", "Total Flow", "Avg Speed"]].copy()

# Make sure numeric columns are numeric
traffic_core["Total Flow"] = pd.to_numeric(traffic_core["Total Flow"], errors="coerce")
traffic_core["Avg Speed"] = pd.to_numeric(traffic_core["Avg Speed"], errors="coerce")
traffic_core["Route"] = pd.to_numeric(traffic_core["Route"], errors="coerce")

print("traffic_core shape:", traffic_core.shape)
display(traffic_core.head())

traffic_core shape: (4114680, 5)


,Timestamp,Station,Route,Total Flow,Avg Speed
0,2024-10-01,308512,50,497.0,64.1
1,2024-10-01,311831,5,27.0,NaN
2,2024-10-01,311832,5,78.0,NaN
3,2024-10-01,311844,5,43.0,NaN
4,2024-10-01,311847,5,73.0,NaN


In [15]:
# Build wide flow table
flow_df = traffic_core.pivot_table(
    index="Timestamp",
    columns="Station",
    values="Total Flow",
    aggfunc="mean"
).sort_index()

# Build wide speed table
speed_df = traffic_core.pivot_table(
    index="Timestamp",
    columns="Station",
    values="Avg Speed",
    aggfunc="mean"
).sort_index()

print("flow_df shape:", flow_df.shape)
print("speed_df shape:", speed_df.shape)

display(flow_df.head())
display(speed_df.head())

flow_df shape: (2208, 1806)
speed_df shape: (2208, 1162)


Station,3001021,3001022,3001101,3001102,3001111,3001121,3003011,3003041,3003044,3003054,...,3423063,3423064,3423065,3423066,3423091,3423094,3900021,3900022,3900023,3900024
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-10-01 00:00:00,121.0,63.0,11.0,113.0,2.0,0.0,45.0,16.0,5.0,1.0,...,2.0,2632.0,2.0,NaN,75.0,0.0,556.0,320.0,546.0,320.0
2024-10-01 01:00:00,98.0,59.0,11.0,84.0,26.0,26.0,31.0,27.0,0.0,1.0,...,3.0,2824.0,1.0,NaN,47.0,20.0,520.0,225.0,505.0,225.0
2024-10-01 02:00:00,99.0,18.0,11.0,94.0,3.0,23.0,35.0,19.0,4.0,0.0,...,3.0,3184.0,4.0,NaN,43.0,27.0,482.0,201.0,519.0,201.0
2024-10-01 03:00:00,97.0,25.0,6.0,75.0,0.0,0.0,25.0,15.0,9.0,0.0,...,2.0,3263.0,1.0,NaN,61.0,15.0,522.0,246.0,583.0,246.0
2024-10-01 04:00:00,188.0,64.0,18.0,161.0,32.0,41.0,47.0,43.0,3.0,0.0,...,18.0,3654.0,6.0,NaN,153.0,79.0,649.0,446.0,915.0,446.0


Station,3001021,3001102,3001111,3001121,3004071,3005031,3005041,3005052,3007031,3007033,...,3423021,3423024,3423061,3423064,3423091,3423094,3900021,3900022,3900023,3900024
Timestamp,,,,,,,,,,,,,,,,,,,,,
2024-10-01 00:00:00,65.3,66.8,66.2,68.2,69.1,64.9,64.9,65.0,66.7,67.4,...,65.8,68.2,66.8,29.1,67.0,68.2,66.5,68.1,66.6,68.1
2024-10-01 01:00:00,65.6,65.1,61.8,61.8,68.5,65.0,64.8,65.1,66.1,67.2,...,65.8,68.2,57.9,32.9,65.7,63.1,66.8,67.8,66.0,67.8
2024-10-01 02:00:00,61.8,63.9,65.9,61.7,68.0,64.9,65.0,64.9,65.9,66.9,...,65.1,68.2,69.6,43.9,66.0,63.4,65.9,67.6,65.4,67.6
2024-10-01 03:00:00,65.1,66.4,68.3,68.2,67.4,64.9,64.9,64.9,66.3,66.5,...,65.6,68.2,66.7,56.4,65.7,61.5,66.2,67.7,66.0,67.7
2024-10-01 04:00:00,65.4,66.8,61.9,61.9,67.0,64.3,64.5,64.8,65.4,66.2,...,66.4,68.2,68.5,58.1,65.6,63.1,66.7,67.9,68.1,67.9


In [16]:
print("Same time index:", flow_df.index.equals(speed_df.index))
print("Same station columns:", flow_df.columns.equals(speed_df.columns))
print("Missing flow values:", flow_df.isna().sum().sum())
print("Missing speed values:", speed_df.isna().sum().sum())

Same time index: True
Same station columns: False
Missing flow values: 176744
Missing speed values: 33610


In [17]:
station_meta_tmp = (
    traffic_core[["Station", "Route"]]
    .drop_duplicates()
    .rename(columns={"Station": "station_id", "Route": "route"})
    .copy()
)

station_meta_tmp["station_id"] = station_meta_tmp["station_id"].astype(str)

print("station_meta_tmp shape:", station_meta_tmp.shape)
display(station_meta_tmp.head())

station_meta_tmp shape: (1896, 2)


,station_id,route
0,308512,50
1,311831,5
2,311832,5
3,311844,5
4,311847,5


In [18]:
xls = pd.ExcelFile("pems_output.xlsx")
print(xls.sheet_names)

['Report Data', 'PeMS Report Description']


In [19]:
meta_raw = pd.read_excel("pems_output.xlsx", sheet_name=0)

print("meta_raw shape:", meta_raw.shape)
print(meta_raw.columns.tolist())
display(meta_raw.head())

meta_raw shape: (1861, 15)
['Fwy', 'District', 'County', 'City', 'CA PM', 'Abs PM', 'Length', 'ID', 'Name', 'Lanes', 'Type', 'Sensor Type', 'HOV', 'MS ID', 'IRM']


,Fwy,District,County,City,CA PM,Abs PM,Length,ID,Name,Lanes,Type,Sensor Type,HOV,MS ID,IRM
0,I5-N,3,Sacramento,NaN,1.919,497.212,4.312,3413014,5NB at Twin Cities Rd,2,Mainline,NaN,No,1,NaN
1,I5-N,3,Sacramento,NaN,2.026,497.319,NaN,3413016,5NB to Twin Cities Rd,4,Off Ramp,NaN,No,1,NaN
2,I5-N,3,Sacramento,NaN,9.498,504.791,3.291,317802,Hood Franklin Rd,2,Mainline,radars,No,1,NaN
3,I5-N,3,Sacramento,NaN,10.942,506.235,3.115,3013091,5NB at Elk Grove HOV,1,HOV,NaN,24H,1,NaN
4,I5-N,3,Sacramento,NaN,11.08,506.373,NaN,311844,Elk Grove Blvd 5NB Slip,2,On Ramp,others,No,1,NaN


In [20]:
# Find common stations
common_stations = flow_df.columns.intersection(speed_df.columns)

print("Common stations:", len(common_stations))

# Filter both
flow_df = flow_df[common_stations].copy()
speed_df = speed_df[common_stations].copy()

# Final check
print("Now same columns:", flow_df.columns.equals(speed_df.columns))
print("New shape:", flow_df.shape)

Common stations: 1162
Now same columns: True
New shape: (2208, 1162)


In [21]:
station_meta = meta_raw[["ID", "Fwy", "Abs PM"]].copy()

station_meta = station_meta.rename(columns={
    "ID": "station_id",
    "Fwy": "route",
    "Abs PM": "abs_pm"
})

station_meta["station_id"] = station_meta["station_id"].astype(str)

# Keep only stations we use
station_meta = station_meta[station_meta["station_id"].isin(flow_df.columns)]

print("station_meta shape:", station_meta.shape)
display(station_meta.head())

station_meta shape: (1154, 3)


,station_id,route,abs_pm
0,3413014,I5-N,497.212
2,317802,I5-N,504.791
3,3013091,I5-N,506.235
5,312134,I5-N,506.373
9,3013103,I5-N,507.464


In [22]:
print("Flow shape:", flow_df.shape)
print("Speed shape:", speed_df.shape)
print("Meta shape:", station_meta.shape)

print("Same index:", flow_df.index.equals(speed_df.index))
print("Same columns:", flow_df.columns.equals(speed_df.columns))
print("Meta covers stations:", set(flow_df.columns).issubset(set(station_meta["station_id"])))

Flow shape: (2208, 1162)
Speed shape: (2208, 1162)
Meta shape: (1154, 3)
Same index: True
Same columns: True
Meta covers stations: False


In [23]:
# Find stations that exist in both
valid_stations = flow_df.columns.intersection(station_meta["station_id"])

print("Valid stations:", len(valid_stations))

# Filter everything
flow_df = flow_df[valid_stations].copy()
speed_df = speed_df[valid_stations].copy()
station_meta = station_meta[station_meta["station_id"].isin(valid_stations)].copy()

# FINAL CHECK
print("Flow shape:", flow_df.shape)
print("Speed shape:", speed_df.shape)
print("Meta shape:", station_meta.shape)

print("Same columns:", flow_df.columns.equals(speed_df.columns))
print("Meta covers stations:",
      set(flow_df.columns).issubset(set(station_meta["station_id"])))

Valid stations: 1154
Flow shape: (2208, 1154)
Speed shape: (2208, 1154)
Meta shape: (1154, 3)
Same columns: True
Meta covers stations: True


# Traffic Forecasting Robustness Study

This notebook builds a leakage-aware traffic forecasting pipeline using PeMS data and compares:

- Random Forest
- LSTM
- CNN-GRU-LSTM
- GraphWaveNet
- GraphWaveNet-GRU
- GraphWaveNet-LSTM
- GraphWaveNet-GRU-LSTM

The study evaluates:
1. Clean forecasting performance
2. Sensor outage robustness
3. Temporal missingness robustness
4. Noise injection robustness

The main goal is to test whether progressive recurrent refinement improves GraphWaveNet, especially at longer horizons and under degraded sensing conditions.

## Configuration

This section defines the forecasting setup:
- 24-hour input window
- 12h, 24h, 48h, 72h prediction horizons
- training parameters
- model size

In [24]:
@dataclass
class Config:
    seed: int = 42
    input_len: int = 24
    horizons: Tuple[int, ...] = (12, 24, 48, 72)
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    batch_size: int = 32
    max_epochs: int = 50
    patience: int = 8
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 5.0
    hidden_dim: int = 64
    gwn_blocks: int = 3
    tcn_kernel: int = 2
    dropout: float = 0.2
    rf_n_estimators: int = 300
    rf_max_depth: Optional[int] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

CFG = Config()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

print(CFG)
print("Device:", CFG.device)

Config(seed=42, input_len=24, horizons=(12, 24, 48, 72), train_ratio=0.7, val_ratio=0.15, batch_size=32, max_epochs=50, patience=8, lr=0.001, weight_decay=1e-05, grad_clip=5.0, hidden_dim=64, gwn_blocks=3, tcn_kernel=2, dropout=0.2, rf_n_estimators=300, rf_max_depth=None, device='cuda')
Device: cuda


## Robustness scenarios

We evaluate the models under four settings:

- Clean: no perturbation
- Sensor outage: full history of a subset of stations is masked
- Temporal missingness: a fraction of input time steps is masked
- Noise injection: Gaussian noise is added to dynamic sensor channels

These perturbations are applied at test time only.

In [25]:
FLOW_SPEED_CHANNELS = [0, 1]

def apply_sensor_outage(x, outage_rate, rng):
    x_new = x.copy()
    N = x.shape[1]
    n_mask = max(1, int(round(outage_rate * N)))
    masked_nodes = np.zeros(N, dtype=bool)
    chosen = rng.choice(N, size=n_mask, replace=False)
    masked_nodes[chosen] = True
    x_new[:, chosen[:, None], FLOW_SPEED_CHANNELS] = 0.0
    return x_new, masked_nodes

def apply_temporal_missingness(x, missing_rate, rng):
    x_new = x.copy()
    L = x.shape[0]
    n_mask = max(1, int(round(missing_rate * L)))
    masked_ts = np.zeros(L, dtype=bool)
    chosen = rng.choice(L, size=n_mask, replace=False)
    masked_ts[chosen] = True
    x_new[chosen[:, None], :, FLOW_SPEED_CHANNELS] = 0.0
    return x_new, masked_ts

def apply_noise_injection(x, noise_std, rng):
    x_new = x.copy()
    noise = rng.normal(loc=0.0, scale=noise_std, size=x_new[:, :, FLOW_SPEED_CHANNELS].shape)
    x_new[:, :, FLOW_SPEED_CHANNELS] += noise.astype(np.float32)
    return x_new

def perturb_window(x, scenario="clean", severity=None, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    info = {}

    if scenario == "clean" or severity is None:
        return x.copy(), info
    if scenario == "sensor_outage":
        x_new, masked_nodes = apply_sensor_outage(x, severity, rng)
        info["masked_nodes"] = masked_nodes
        return x_new, info
    if scenario == "temporal_missingness":
        x_new, masked_ts = apply_temporal_missingness(x, severity, rng)
        info["masked_timesteps"] = masked_ts
        return x_new, info
    if scenario == "noise_injection":
        x_new = apply_noise_injection(x, severity, rng)
        return x_new, info

    raise ValueError(f"Unknown scenario: {scenario}")

## Clean benchmark

This section runs all models under normal conditions and creates the main MAE and RMSE tables.

In [26]:
# Check missingness before cleaning
print("Initial flow missing:", flow_df.isna().sum().sum())
print("Initial speed missing:", speed_df.isna().sum().sum())

flow_missing_rate = flow_df.isna().mean()
speed_missing_rate = speed_df.isna().mean()

missing_summary = pd.DataFrame({
    "flow_missing_rate": flow_missing_rate,
    "speed_missing_rate": speed_missing_rate
})

display(missing_summary.describe())

Initial flow missing: 31366
Initial speed missing: 31366


,flow_missing_rate,speed_missing_rate
count,1154.000000,1154.000000
mean,0.012310,0.012310
std,0.094875,0.094875
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,0.750000,0.750000


In [27]:
# Drop only extreme cases (>50% missing)
threshold = 0.5

valid_station_mask = (
    (flow_df.isna().mean() <= threshold) &
    (speed_df.isna().mean() <= threshold)
)

valid_stations = flow_df.columns[valid_station_mask]

flow_df = flow_df[valid_stations].copy()
speed_df = speed_df[valid_stations].copy()
station_meta = station_meta[station_meta["station_id"].isin(valid_stations)].copy()

print("Stations after filtering:", len(valid_stations))

Stations after filtering: 1135


In [28]:
flow_df = flow_df.interpolate(method="time").ffill().bfill()
speed_df = speed_df.interpolate(method="time").ffill().bfill()

print("Flow NaNs:", flow_df.isna().sum().sum())
print("Speed NaNs:", speed_df.isna().sum().sum())

Flow NaNs: 0
Speed NaNs: 0


In [29]:
print("Same index:", flow_df.index.equals(speed_df.index))
print("Same columns:", flow_df.columns.equals(speed_df.columns))
print("Meta covers stations:", set(flow_df.columns).issubset(set(station_meta["station_id"])))
print("Any NaNs left:", flow_df.isna().any().any(), speed_df.isna().any().any())

Same index: True
Same columns: True
Meta covers stations: True
Any NaNs left: False False


In [30]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 0) CONFIG
# ============================================================

@dataclass
class Config:
    seed: int = 42
    input_len: int = 24
    horizons: Tuple[int, ...] = (12, 24, 48, 72)
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    batch_size: int = 32
    max_epochs: int = 50
    patience: int = 8
    lr: float = 1e-3
    weight_decay: float = 1e-5
    grad_clip: float = 5.0
    hidden_dim: int = 64
    gwn_blocks: int = 3
    tcn_kernel: int = 2
    dropout: float = 0.2
    rf_n_estimators: int = 300
    rf_max_depth: Optional[int] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


CFG = Config()


# ============================================================
# 1) REPRODUCIBILITY
# ============================================================

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG.seed)


# ============================================================
# 2) DATA PREP
# ============================================================

def make_calendar_features(index: pd.DatetimeIndex) -> pd.DataFrame:
    hour = index.hour.values
    dow = index.dayofweek.values
    return pd.DataFrame(
        {
            "sin_hour": np.sin(2 * np.pi * hour / 24.0),
            "cos_hour": np.cos(2 * np.pi * hour / 24.0),
            "sin_dow": np.sin(2 * np.pi * dow / 7.0),
            "cos_dow": np.cos(2 * np.pi * dow / 7.0),
        },
        index=index,
    )


def train_only_standardize(
    df: pd.DataFrame, train_end_idx: int, eps: float = 1e-6
) -> Tuple[pd.DataFrame, pd.Series, pd.Series]:
    train_part = df.iloc[:train_end_idx]
    mu = train_part.mean(axis=0)
    sigma = train_part.std(axis=0).replace(0, 1.0)
    z = (df - mu) / (sigma + eps)
    return z, mu, sigma


def build_feature_tensor(
    flow_df: pd.DataFrame,
    speed_df: pd.DataFrame,
    cfg: Config,
) -> Tuple[np.ndarray, np.ndarray, pd.DatetimeIndex, Dict[str, pd.Series]]:
    """
    Returns:
        X_all: [T, N, F] standardized flow/speed + repeated calendar
        y_flow_original: [T, N] original flow
        time_index
        scalers: flow_mu, flow_sigma
    """
    assert flow_df.index.equals(speed_df.index), "flow_df and speed_df must share the same DatetimeIndex"
    assert flow_df.columns.equals(speed_df.columns), "flow_df and speed_df must share the same station columns"

    T = len(flow_df)
    train_end = int(T * cfg.train_ratio)

    flow_z, flow_mu, flow_sigma = train_only_standardize(flow_df, train_end)
    speed_z, speed_mu, speed_sigma = train_only_standardize(speed_df, train_end)
    cal = make_calendar_features(flow_df.index)

    flow_arr = flow_z.values.astype(np.float32)   # [T, N]
    speed_arr = speed_z.values.astype(np.float32) # [T, N]
    cal_arr = cal.values.astype(np.float32)       # [T, 4]

    N = flow_arr.shape[1]
    cal_rep = np.repeat(cal_arr[:, None, :], repeats=N, axis=1)  # [T, N, 4]

    X_all = np.concatenate(
        [
            flow_arr[:, :, None],
            speed_arr[:, :, None],
            cal_rep,
        ],
        axis=-1,
    ).astype(np.float32)  # [T, N, 6]

    y_flow_original = flow_df.values.astype(np.float32)
    scalers = {
        "flow_mu": flow_mu.astype(np.float32),
        "flow_sigma": flow_sigma.astype(np.float32),
        "speed_mu": speed_mu.astype(np.float32),
        "speed_sigma": speed_sigma.astype(np.float32),
    }
    return X_all, y_flow_original, flow_df.index, scalers


def build_supports_from_metadata(
    station_meta: pd.DataFrame,
    station_ids: Sequence,
    route_col: str = "route",
    postmile_col: str = "abs_pm",
    k_neighbors: int = 2,
    sigma: Optional[float] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build fixed metadata-driven adjacency from route + postmile.
    station_meta must have columns: station_id, route, abs_pm
    """
    meta = station_meta.copy()
    if "station_id" not in meta.columns:
        raise ValueError("station_meta must contain a 'station_id' column")

    meta = meta.set_index("station_id").loc[list(station_ids)].reset_index()
    N = len(meta)
    A = np.zeros((N, N), dtype=np.float32)

    for route, grp in meta.groupby(route_col):
        grp = grp.sort_values(postmile_col).reset_index()
        idxs = grp["index"].tolist()
        pms = grp[postmile_col].astype(float).values

        for local_i, i in enumerate(idxs):
            dist = np.abs(pms - pms[local_i])
            order = np.argsort(dist)
            neigh = [idxs[j] for j in order[1 : k_neighbors + 1]]
            for j in neigh:
                d = abs(meta.loc[i, postmile_col] - meta.loc[j, postmile_col])
                A[i, j] = d

    nonzero = A[A > 0]
    if sigma is None:
        sigma = float(np.median(nonzero)) if len(nonzero) else 1.0

    W = np.zeros_like(A, dtype=np.float32)
    for i in range(N):
        for j in range(N):
            if A[i, j] > 0:
                W[i, j] = math.exp(-(A[i, j] ** 2) / (sigma ** 2 + 1e-8))
    np.fill_diagonal(W, 1.0)

    W = np.maximum(W, W.T)

    D = W.sum(axis=1, keepdims=True) + 1e-8
    A_rw = W / D
    A_rev_rw = W.T / (W.T.sum(axis=1, keepdims=True) + 1e-8)

    return W.astype(np.float32), A_rw.astype(np.float32), A_rev_rw.astype(np.float32)


def make_valid_window_starts(
    time_index: pd.DatetimeIndex,
    cfg: Config,
) -> Dict[str, np.ndarray]:
    """
    Split by target period, not input period.
    """
    T = len(time_index)
    L = cfg.input_len
    Hmax = max(cfg.horizons)

    train_end = int(T * cfg.train_ratio)
    val_end = int(T * (cfg.train_ratio + cfg.val_ratio))

    starts = []
    for tau in range(T - L - Hmax + 1):
        target_start = tau + L
        target_end = tau + L + Hmax - 1
        starts.append((tau, target_start, target_end))

    train_starts, val_starts, test_starts = [], [], []
    for tau, target_start, target_end in starts:
        if target_end < train_end:
            train_starts.append(tau)
        elif target_start >= train_end and target_end < val_end:
            val_starts.append(tau)
        elif target_start >= val_end:
            test_starts.append(tau)

    return {
        "train": np.array(train_starts, dtype=np.int64),
        "val": np.array(val_starts, dtype=np.int64),
        "test": np.array(test_starts, dtype=np.int64),
    }


# ============================================================
# 3) ROBUSTNESS SCENARIOS
# ============================================================

FLOW_SPEED_CHANNELS = [0, 1]  # only dynamic sensor channels


def apply_sensor_outage(
    x: np.ndarray,
    outage_rate: float,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    x: [L, N, F]
    Masks full history for a subset of nodes.
    Returns:
        x_new
        masked_nodes: [N] bool
    """
    x_new = x.copy()
    N = x.shape[1]
    n_mask = max(1, int(round(outage_rate * N)))
    masked_nodes = np.zeros(N, dtype=bool)
    chosen = rng.choice(N, size=n_mask, replace=False)
    masked_nodes[chosen] = True
    x_new[:, chosen[:, None], FLOW_SPEED_CHANNELS] = 0.0
    return x_new, masked_nodes


def apply_temporal_missingness(
    x: np.ndarray,
    missing_rate: float,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Randomly drop a fraction of time steps across the entire input window.
    Only flow/speed channels are zeroed; calendar is retained.
    Returns:
        x_new
        masked_timesteps: [L] bool
    """
    x_new = x.copy()
    L = x.shape[0]
    n_mask = max(1, int(round(missing_rate * L)))
    masked_ts = np.zeros(L, dtype=bool)
    chosen = rng.choice(L, size=n_mask, replace=False)
    masked_ts[chosen] = True
    x_new[chosen[:, None], :, FLOW_SPEED_CHANNELS] = 0.0
    return x_new, masked_ts


def apply_noise_injection(
    x: np.ndarray,
    noise_std: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """
    Add Gaussian noise to standardized flow/speed channels only.
    """
    x_new = x.copy()
    noise = rng.normal(loc=0.0, scale=noise_std, size=x_new[:, :, FLOW_SPEED_CHANNELS].shape)
    x_new[:, :, FLOW_SPEED_CHANNELS] += noise.astype(np.float32)
    return x_new


def perturb_window(
    x: np.ndarray,
    scenario: str = "clean",
    severity: Optional[float] = None,
    rng: Optional[np.random.Generator] = None,
) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
    rng = np.random.default_rng() if rng is None else rng
    info = {}

    if scenario == "clean" or severity is None:
        return x.copy(), info
    if scenario == "sensor_outage":
        x_new, masked_nodes = apply_sensor_outage(x, severity, rng)
        info["masked_nodes"] = masked_nodes
        return x_new, info
    if scenario == "temporal_missingness":
        x_new, masked_ts = apply_temporal_missingness(x, severity, rng)
        info["masked_timesteps"] = masked_ts
        return x_new, info
    if scenario == "noise_injection":
        x_new = apply_noise_injection(x, severity, rng)
        return x_new, info

    raise ValueError(f"Unknown scenario: {scenario}")


# ============================================================
# 4) PYTORCH DATASET
# ============================================================

class TrafficWindowDataset(Dataset):
    def __init__(
        self,
        X_all: np.ndarray,                  # [T, N, F]
        y_flow_original: np.ndarray,        # [T, N] original unit
        flow_mu: pd.Series,
        flow_sigma: pd.Series,
        time_index: pd.DatetimeIndex,
        starts: np.ndarray,
        cfg: Config,
        scenario: str = "clean",
        severity: Optional[float] = None,
        seed: int = 42,
    ):
        self.X_all = X_all
        self.y_flow_original = y_flow_original
        self.flow_mu = flow_mu.values.astype(np.float32)
        self.flow_sigma = flow_sigma.values.astype(np.float32)
        self.time_index = time_index
        self.starts = starts
        self.cfg = cfg
        self.scenario = scenario
        self.severity = severity
        self.rng = np.random.default_rng(seed)
        self.horizons = np.array(cfg.horizons, dtype=np.int64)

    def __len__(self) -> int:
        return len(self.starts)

    def __getitem__(self, idx: int):
        tau = int(self.starts[idx])
        L = self.cfg.input_len

        x = self.X_all[tau : tau + L].copy()  # [L, N, F]
        x, info = perturb_window(
            x, scenario=self.scenario, severity=self.severity, rng=self.rng
        )

        target_times = tau + L + self.horizons - 1
        y_orig = self.y_flow_original[target_times, :]  # [K, N]
        y_z = (y_orig - self.flow_mu[None, :]) / (self.flow_sigma[None, :] + 1e-6)

        future_cal = make_calendar_features(self.time_index[target_times]).values.astype(np.float32)  # [K, 4]

        item = {
            "x": torch.tensor(x, dtype=torch.float32),
            "future_cal": torch.tensor(future_cal, dtype=torch.float32),
            "y_z": torch.tensor(y_z, dtype=torch.float32),
            "y_orig": torch.tensor(y_orig, dtype=torch.float32),
        }

        if "masked_nodes" in info:
            item["masked_nodes"] = torch.tensor(info["masked_nodes"], dtype=torch.bool)
        return item


# ============================================================
# 5) MODEL COMPONENTS
# ============================================================

class HorizonHead(nn.Module):
    def __init__(self, hidden_dim: int, cal_dim: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim + cal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, node_repr: torch.Tensor, future_cal: torch.Tensor) -> torch.Tensor:
        """
        node_repr: [B, N, C]
        future_cal: [B, K, 4]
        Returns:
            yhat: [B, K, N]
        """
        B, N, C = node_repr.shape
        K = future_cal.shape[1]

        node_expand = node_repr[:, None, :, :].expand(B, K, N, C)
        cal_expand = future_cal[:, :, None, :].expand(B, K, N, future_cal.shape[-1])
        z = torch.cat([node_expand, cal_expand], dim=-1)
        yhat = self.net(z).squeeze(-1)
        return yhat


class DiffusionGraphConv(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.self_lin = nn.Linear(channels, channels)
        self.fwd_lin = nn.Linear(channels, channels)
        self.rev_lin = nn.Linear(channels, channels)

    def forward(self, x: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        """
        x: [B, T, N, C]
        A_*: [N, N]
        """
        agg_fwd = torch.einsum("ij,btjc->btic", A_fwd, x)
        agg_rev = torch.einsum("ij,btjc->btic", A_rev, x)
        return self.self_lin(x) + self.fwd_lin(agg_fwd) + self.rev_lin(agg_rev)


class GraphWaveNetBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int, dilation: int, dropout: float):
        super().__init__()
        pad = (kernel_size - 1) * dilation
        self.filter_conv = nn.Conv2d(
            channels, channels, kernel_size=(kernel_size, 1), dilation=(dilation, 1), padding=(pad, 0)
        )
        self.gate_conv = nn.Conv2d(
            channels, channels, kernel_size=(kernel_size, 1), dilation=(dilation, 1), padding=(pad, 0)
        )
        self.graph_conv = DiffusionGraphConv(channels)
        self.res_proj = nn.Linear(channels, channels)
        self.skip_proj = nn.Linear(channels, channels)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(channels)

    def forward(self, x: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x: [B, T, N, C]
        """
        residual = x
        z = x.permute(0, 3, 1, 2)  # [B, C, T, N]
        z = torch.tanh(self.filter_conv(z)) * torch.sigmoid(self.gate_conv(z))
        z = z[:, :, : x.shape[1], :]  # crop to original length
        z = z.permute(0, 2, 3, 1)     # [B, T, N, C]

        z = self.graph_conv(z, A_fwd, A_rev)
        z = self.dropout(z)

        out = self.norm(self.res_proj(z) + residual)
        skip = self.skip_proj(z)
        return out, skip


class GraphWaveNetEncoder(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        num_blocks: int,
        kernel_size: int,
        dropout: float,
    ):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.ModuleList(
            [
                GraphWaveNetBlock(hidden_dim, kernel_size=kernel_size, dilation=2 ** i, dropout=dropout)
                for i in range(num_blocks)
            ]
        )
        self.out_norm = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        """
        x: [B, L, N, F]
        returns: [B, L, N, C]
        """
        h = self.input_proj(x)
        skip_sum = 0.0
        for block in self.blocks:
            h, skip = block(h, A_fwd, A_rev)
            skip_sum = skip_sum + skip
        return self.out_norm(skip_sum)


class BaseForecastModel(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.head = HorizonHead(hidden_dim)

    def predict_from_node_repr(self, node_repr: torch.Tensor, future_cal: torch.Tensor) -> torch.Tensor:
        return self.head(node_repr, future_cal)


class PlainLSTMModel(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int):
        super().__init__(hidden_dim)
        self.lstm = nn.LSTM(input_size=in_dim, hidden_size=hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, **kwargs) -> torch.Tensor:
        B, L, N, F = x.shape
        x_bn = x.permute(0, 2, 1, 3).reshape(B * N, L, F)
        _, (h_n, _) = self.lstm(x_bn)
        node_repr = h_n[-1].reshape(B, N, -1)
        return self.predict_from_node_repr(node_repr, future_cal)


class CNNGRULSTMModel(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int, dropout: float):
        super().__init__(hidden_dim)
        self.conv = nn.Conv1d(in_channels=in_dim, out_channels=hidden_dim, kernel_size=3, padding=2)
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, **kwargs) -> torch.Tensor:
        B, L, N, F = x.shape
        x_bn = x.permute(0, 2, 1, 3).reshape(B * N, L, F)         # [B*N, L, F]
        z = x_bn.transpose(1, 2)                                  # [B*N, F, L]
        z = self.conv(z)[:, :, :L]                                # [B*N, C, L]
        z = torch.relu(z.transpose(1, 2))                         # [B*N, L, C]
        z, _ = self.gru(z)
        z, (h_n, _) = self.lstm(self.dropout(z))
        node_repr = h_n[-1].reshape(B, N, -1)
        return self.predict_from_node_repr(node_repr, future_cal)


class GraphWaveNetOnly(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int, cfg: Config):
        super().__init__(hidden_dim)
        self.encoder = GraphWaveNetEncoder(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            num_blocks=cfg.gwn_blocks,
            kernel_size=cfg.tcn_kernel,
            dropout=cfg.dropout,
        )

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x, A_fwd, A_rev)
        node_repr = h[:, -1, :, :]  # last time step
        return self.predict_from_node_repr(node_repr, future_cal)


class GraphWaveNetGRU(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int, cfg: Config):
        super().__init__(hidden_dim)
        self.encoder = GraphWaveNetEncoder(in_dim, hidden_dim, cfg.gwn_blocks, cfg.tcn_kernel, cfg.dropout)
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x, A_fwd, A_rev)                          # [B, L, N, C]
        B, L, N, C = h.shape
        h_bn = h.permute(0, 2, 1, 3).reshape(B * N, L, C)
        _, h_n = self.gru(h_bn)
        node_repr = h_n[-1].reshape(B, N, C)
        return self.predict_from_node_repr(node_repr, future_cal)


class GraphWaveNetLSTM(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int, cfg: Config):
        super().__init__(hidden_dim)
        self.encoder = GraphWaveNetEncoder(in_dim, hidden_dim, cfg.gwn_blocks, cfg.tcn_kernel, cfg.dropout)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x, A_fwd, A_rev)
        B, L, N, C = h.shape
        h_bn = h.permute(0, 2, 1, 3).reshape(B * N, L, C)
        _, (h_n, _) = self.lstm(h_bn)
        node_repr = h_n[-1].reshape(B, N, C)
        return self.predict_from_node_repr(node_repr, future_cal)


class GraphWaveNetGRULSTM(BaseForecastModel):
    def __init__(self, in_dim: int, hidden_dim: int, cfg: Config):
        super().__init__(hidden_dim)
        self.encoder = GraphWaveNetEncoder(in_dim, hidden_dim, cfg.gwn_blocks, cfg.tcn_kernel, cfg.dropout)
        self.gru = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.lstm = nn.LSTM(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor, A_fwd: torch.Tensor, A_rev: torch.Tensor) -> torch.Tensor:
        h = self.encoder(x, A_fwd, A_rev)
        B, L, N, C = h.shape
        h_bn = h.permute(0, 2, 1, 3).reshape(B * N, L, C)
        z, _ = self.gru(h_bn)
        _, (h_n, _) = self.lstm(z)
        node_repr = h_n[-1].reshape(B, N, C)
        return self.predict_from_node_repr(node_repr, future_cal)


# ============================================================
# 6) TRAIN / EVAL
# ============================================================

def inverse_scale_flow(
    y_z: np.ndarray, flow_mu: pd.Series, flow_sigma: pd.Series
) -> np.ndarray:
    mu = flow_mu.values[None, None, :]
    sigma = flow_sigma.values[None, None, :]
    return y_z * (sigma + 1e-6) + mu


def batch_inverse_scale_flow(
    y_z: torch.Tensor, flow_mu: pd.Series, flow_sigma: pd.Series
) -> torch.Tensor:
    mu = torch.tensor(flow_mu.values, dtype=y_z.dtype, device=y_z.device)[None, None, :]
    sigma = torch.tensor(flow_sigma.values, dtype=y_z.dtype, device=y_z.device)[None, None, :]
    return y_z * (sigma + 1e-6) + mu


def make_loaders(
    X_all: np.ndarray,
    y_flow_original: np.ndarray,
    time_index: pd.DatetimeIndex,
    scalers: Dict[str, pd.Series],
    starts_dict: Dict[str, np.ndarray],
    cfg: Config,
    scenario: str = "clean",
    severity: Optional[float] = None,
):
    train_ds = TrafficWindowDataset(
        X_all, y_flow_original, scalers["flow_mu"], scalers["flow_sigma"], time_index,
        starts_dict["train"], cfg, scenario="clean", severity=None, seed=cfg.seed
    )
    val_ds = TrafficWindowDataset(
        X_all, y_flow_original, scalers["flow_mu"], scalers["flow_sigma"], time_index,
        starts_dict["val"], cfg, scenario="clean", severity=None, seed=cfg.seed + 1
    )
    test_ds = TrafficWindowDataset(
        X_all, y_flow_original, scalers["flow_mu"], scalers["flow_sigma"], time_index,
        starts_dict["test"], cfg, scenario=scenario, severity=severity, seed=cfg.seed + 2
    )
    return (
        DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True),
        DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False),
        DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False),
    )


def compute_metrics_by_horizon(y_true: np.ndarray, y_pred: np.ndarray, horizons: Sequence[int]) -> Dict[str, Dict[int, float]]:
    """
    y_true, y_pred: [num_windows, K, N] in original units
    """
    out = {"MAE": {}, "RMSE": {}}
    for k, h in enumerate(horizons):
        yt = y_true[:, k, :].reshape(-1)
        yp = y_pred[:, k, :].reshape(-1)
        out["MAE"][int(h)] = float(mean_absolute_error(yt, yp))
        out["RMSE"][int(h)] = float(np.sqrt(mean_squared_error(yt, yp)))
    return out


def evaluate_masked_nodes_only(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    masked_nodes: np.ndarray,
    horizons: Sequence[int],
) -> Dict[str, Dict[int, float]]:
    """
    masked_nodes: [num_windows, N] boolean
    """
    out = {"MAE": {}, "RMSE": {}}
    for k, h in enumerate(horizons):
        errs = y_pred[:, k, :] - y_true[:, k, :]
        mask = masked_nodes
        masked_errs = errs[mask]
        masked_true = y_true[:, k, :][mask]
        masked_pred = y_pred[:, k, :][mask]
        out["MAE"][int(h)] = float(mean_absolute_error(masked_true, masked_pred))
        out["RMSE"][int(h)] = float(np.sqrt(mean_squared_error(masked_true, masked_pred)))
    return out


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: Optional[torch.optim.Optimizer],
    device: str,
    A_fwd: Optional[torch.Tensor] = None,
    A_rev: Optional[torch.Tensor] = None,
) -> float:
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    loss_fn = nn.SmoothL1Loss()
    total_loss, total_count = 0.0, 0

    for batch in loader:
        x = batch["x"].to(device)
        future_cal = batch["future_cal"].to(device)
        y_z = batch["y_z"].to(device)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            if A_fwd is None:
                y_hat = model(x, future_cal)
            else:
                y_hat = model(x, future_cal, A_fwd=A_fwd, A_rev=A_rev)

            loss = loss_fn(y_hat, y_z)

            if train_mode:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.grad_clip)
                optimizer.step()

        bs = x.shape[0]
        total_loss += loss.item() * bs
        total_count += bs

    return total_loss / max(total_count, 1)


def fit_neural_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
    A_fwd: Optional[torch.Tensor] = None,
    A_rev: Optional[torch.Tensor] = None,
) -> nn.Module:
    model = model.to(cfg.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    best_state = None
    best_val = float("inf")
    patience_count = 0

    for epoch in range(cfg.max_epochs):
        train_loss = run_epoch(model, train_loader, optimizer, cfg.device, A_fwd=A_fwd, A_rev=A_rev)
        val_loss = run_epoch(model, val_loader, None, cfg.device, A_fwd=A_fwd, A_rev=A_rev)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg.patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def predict_neural_model(
    model: nn.Module,
    loader: DataLoader,
    cfg: Config,
    flow_mu: pd.Series,
    flow_sigma: pd.Series,
    A_fwd: Optional[torch.Tensor] = None,
    A_rev: Optional[torch.Tensor] = None,
) -> Dict[str, np.ndarray]:
    model.eval()
    ys_true, ys_pred, masked_nodes_all = [], [], []

    with torch.no_grad():
        for batch in loader:
            x = batch["x"].to(cfg.device)
            future_cal = batch["future_cal"].to(cfg.device)

            if A_fwd is None:
                y_hat_z = model(x, future_cal)
            else:
                y_hat_z = model(x, future_cal, A_fwd=A_fwd, A_rev=A_rev)

            y_hat = batch_inverse_scale_flow(y_hat_z, flow_mu, flow_sigma).cpu().numpy()
            y_true = batch["y_orig"].numpy()

            ys_true.append(y_true)
            ys_pred.append(y_hat)

            if "masked_nodes" in batch:
                masked_nodes_all.append(batch["masked_nodes"].numpy())

    out = {
        "y_true": np.concatenate(ys_true, axis=0),
        "y_pred": np.concatenate(ys_pred, axis=0),
    }
    if masked_nodes_all:
        out["masked_nodes"] = np.concatenate(masked_nodes_all, axis=0)
    return out


# ============================================================
# 7) RANDOM FOREST BASELINE
# ============================================================

def build_rf_features_for_station(
    X_all: np.ndarray,
    time_index: pd.DatetimeIndex,
    starts: np.ndarray,
    station_idx: int,
    cfg: Config,
    scenario: str = "clean",
    severity: Optional[float] = None,
    seed: int = 42,
) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    X_feats, Y = [], []

    for tau in starts:
        x = X_all[tau : tau + cfg.input_len, station_idx : station_idx + 1, :].copy()  # [L,1,F]
        x, _ = perturb_window(x, scenario=scenario, severity=severity, rng=rng)
        x = x[:, 0, :].reshape(-1)  # flatten [L*F]

        target_times = tau + cfg.input_len + np.array(cfg.horizons) - 1
        future_cal = make_calendar_features(time_index[target_times]).values.reshape(-1)

        feat = np.concatenate([x, future_cal], axis=0)
        target = X_all[target_times, station_idx, 0]  # standardized flow channel
        X_feats.append(feat)
        Y.append(target)

    return np.stack(X_feats).astype(np.float32), np.stack(Y).astype(np.float32)


def fit_predict_random_forest(
    X_all: np.ndarray,
    y_flow_original: np.ndarray,
    time_index: pd.DatetimeIndex,
    scalers: Dict[str, pd.Series],
    starts_dict: Dict[str, np.ndarray],
    cfg: Config,
    scenario: str = "clean",
    severity: Optional[float] = None,
) -> Dict[str, np.ndarray]:
    N = X_all.shape[1]
    horizons = np.array(cfg.horizons)

    num_test = len(starts_dict["test"])
    K = len(horizons)
    y_true = np.zeros((num_test, K, N), dtype=np.float32)
    y_pred = np.zeros((num_test, K, N), dtype=np.float32)

    for n in range(N):
        X_train, Y_train = build_rf_features_for_station(
            X_all, time_index, starts_dict["train"], n, cfg, scenario="clean", severity=None, seed=cfg.seed
        )
        X_test, _ = build_rf_features_for_station(
            X_all, time_index, starts_dict["test"], n, cfg, scenario=scenario, severity=severity, seed=cfg.seed + 1
        )

        model = MultiOutputRegressor(
            RandomForestRegressor(
                n_estimators=cfg.rf_n_estimators,
                max_depth=cfg.rf_max_depth,
                random_state=cfg.seed,
                n_jobs=-1,
            )
        )
        model.fit(X_train, Y_train)
        pred_z = model.predict(X_test)

        mu = scalers["flow_mu"].iloc[n]
        sigma = scalers["flow_sigma"].iloc[n]
        pred_orig = pred_z * (sigma + 1e-6) + mu

        target_times = starts_dict["test"][:, None] + cfg.input_len + horizons[None, :] - 1
        y_true[:, :, n] = y_flow_original[target_times, n]
        y_pred[:, :, n] = pred_orig

    return {"y_true": y_true, "y_pred": y_pred}


# ============================================================
# 8) EXPERIMENT RUNNER
# ============================================================

def make_model_dict(cfg: Config, in_dim: int) -> Dict[str, nn.Module]:
    return {
        "LSTM": PlainLSTMModel(in_dim=in_dim, hidden_dim=cfg.hidden_dim),
        "CNN_GRU_LSTM": CNNGRULSTMModel(in_dim=in_dim, hidden_dim=cfg.hidden_dim, dropout=cfg.dropout),
        "GraphWaveNet": GraphWaveNetOnly(in_dim=in_dim, hidden_dim=cfg.hidden_dim, cfg=cfg),
        "GraphWaveNet_GRU": GraphWaveNetGRU(in_dim=in_dim, hidden_dim=cfg.hidden_dim, cfg=cfg),
        "GraphWaveNet_LSTM": GraphWaveNetLSTM(in_dim=in_dim, hidden_dim=cfg.hidden_dim, cfg=cfg),
        "GraphWaveNet_GRU_LSTM": GraphWaveNetGRULSTM(in_dim=in_dim, hidden_dim=cfg.hidden_dim, cfg=cfg),
    }


def run_experiment_suite(
    flow_df: pd.DataFrame,
    speed_df: pd.DataFrame,
    station_meta: pd.DataFrame,
    cfg: Config,
    scenario: str = "clean",
    severity: Optional[float] = None,
) -> Dict[str, Dict]:
    X_all, y_flow_original, time_index, scalers = build_feature_tensor(flow_df, speed_df, cfg)
    starts_dict = make_valid_window_starts(time_index, cfg)

    _, A_rw_np, A_rev_rw_np = build_supports_from_metadata(
        station_meta=station_meta,
        station_ids=flow_df.columns,
        route_col="route",
        postmile_col="abs_pm",
        k_neighbors=2,
    )

    A_fwd = torch.tensor(A_rw_np, dtype=torch.float32, device=cfg.device)
    A_rev = torch.tensor(A_rev_rw_np, dtype=torch.float32, device=cfg.device)

    train_loader, val_loader, test_loader = make_loaders(
        X_all, y_flow_original, time_index, scalers, starts_dict, cfg,
        scenario=scenario, severity=severity,
    )

    results = {}

    rf_out = fit_predict_random_forest(
        X_all, y_flow_original, time_index, scalers, starts_dict, cfg,
        scenario=scenario, severity=severity,
    )
    results["RandomForest"] = {
        "predictions": rf_out,
        "metrics": compute_metrics_by_horizon(rf_out["y_true"], rf_out["y_pred"], cfg.horizons),
    }

    models = make_model_dict(cfg, in_dim=X_all.shape[-1])

    for name, model in models.items():
        is_graph = "GraphWaveNet" in name
        model = fit_neural_model(
            model, train_loader, val_loader, cfg,
            A_fwd=A_fwd if is_graph else None,
            A_rev=A_rev if is_graph else None,
        )
        out = predict_neural_model(
            model, test_loader, cfg, scalers["flow_mu"], scalers["flow_sigma"],
            A_fwd=A_fwd if is_graph else None,
            A_rev=A_rev if is_graph else None,
        )
        res = {
            "predictions": out,
            "metrics": compute_metrics_by_horizon(out["y_true"], out["y_pred"], cfg.horizons),
        }
        if "masked_nodes" in out:
            res["masked_only_metrics"] = evaluate_masked_nodes_only(
                out["y_true"], out["y_pred"], out["masked_nodes"], cfg.horizons
            )
        results[name] = res

    return results


# ============================================================
# 9) TABLE HELPERS
# ============================================================

def metrics_to_table(results: Dict[str, Dict], metric_name: str = "MAE") -> pd.DataFrame:
    rows = {}
    for model_name, payload in results.items():
        rows[model_name] = payload["metrics"][metric_name]
    df = pd.DataFrame(rows).T
    df = df[[h for h in CFG.horizons if h in df.columns]]
    return df


def relative_improvement_vs_base(
    results: Dict[str, Dict],
    base_model: str = "GraphWaveNet",
    metric_name: str = "MAE",
) -> pd.DataFrame:
    base = pd.Series(results[base_model]["metrics"][metric_name])
    rows = {}
    for model_name, payload in results.items():
        if model_name == base_model:
            continue
        vals = pd.Series(payload["metrics"][metric_name])
        rows[model_name] = 100.0 * (base - vals) / base
    return pd.DataFrame(rows).T

In [31]:
def prepare_experiment_objects(flow_df, speed_df, station_meta, cfg, scenario="clean", severity=None):
    X_all, y_flow_original, time_index, scalers = build_feature_tensor(flow_df, speed_df, cfg)
    starts_dict = make_valid_window_starts(time_index, cfg)

    _, A_rw_np, A_rev_rw_np = build_supports_from_metadata(
        station_meta=station_meta,
        station_ids=flow_df.columns,
        route_col="route",
        postmile_col="abs_pm",
        k_neighbors=2,
    )

    A_fwd = torch.tensor(A_rw_np, dtype=torch.float32, device=cfg.device)
    A_rev = torch.tensor(A_rev_rw_np, dtype=torch.float32, device=cfg.device)

    train_loader, val_loader, test_loader = make_loaders(
        X_all, y_flow_original, time_index, scalers, starts_dict, cfg,
        scenario=scenario, severity=severity,
    )

    return {
        "X_all": X_all,
        "y_flow_original": y_flow_original,
        "time_index": time_index,
        "scalers": scalers,
        "starts_dict": starts_dict,
        "A_fwd": A_fwd,
        "A_rev": A_rev,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }

In [32]:
def run_single_model(
    model_name,
    prepared,
    cfg,
    scenario="clean",
    severity=None,
):
    scalers = prepared["scalers"]

    if model_name == "RandomForest":
        out = fit_predict_random_forest(
            prepared["X_all"],
            prepared["y_flow_original"],
            prepared["time_index"],
            scalers,
            prepared["starts_dict"],
            cfg,
            scenario=scenario,
            severity=severity,
        )
        result = {
            "model_name": model_name,
            "scenario": scenario,
            "severity": severity,
            "predictions": out,
            "metrics": compute_metrics_by_horizon(out["y_true"], out["y_pred"], cfg.horizons),
        }
        return result

    model_dict = make_model_dict(cfg, in_dim=prepared["X_all"].shape[-1])
    model = model_dict[model_name]

    is_graph = "GraphWaveNet" in model_name

    model = fit_neural_model(
        model,
        prepared["train_loader"],
        prepared["val_loader"],
        cfg,
        A_fwd=prepared["A_fwd"] if is_graph else None,
        A_rev=prepared["A_rev"] if is_graph else None,
    )

    out = predict_neural_model(
        model,
        prepared["test_loader"],
        cfg,
        scalers["flow_mu"],
        scalers["flow_sigma"],
        A_fwd=prepared["A_fwd"] if is_graph else None,
        A_rev=prepared["A_rev"] if is_graph else None,
    )

    result = {
        "model_name": model_name,
        "scenario": scenario,
        "severity": severity,
        "predictions": out,
        "metrics": compute_metrics_by_horizon(out["y_true"], out["y_pred"], cfg.horizons),
    }

    if "masked_nodes" in out:
        result["masked_only_metrics"] = evaluate_masked_nodes_only(
            out["y_true"], out["y_pred"], out["masked_nodes"], cfg.horizons
        )

    return result

In [33]:
import pickle
from pathlib import Path

RESULTS_DIR = Path("saved_results")
RESULTS_DIR.mkdir(exist_ok=True)

def save_result(result, filename):
    filepath = RESULTS_DIR / filename
    with open(filepath, "wb") as f:
        pickle.dump(result, f)
    print("Saved:", filepath)

def load_result(filename):
    filepath = RESULTS_DIR / filename
    with open(filepath, "rb") as f:
        return pickle.load(f)

In [34]:
prepared_clean = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="clean",
    severity=None,
)

print("Prepared clean experiment objects.")

Prepared clean experiment objects.


In [ ]:
rf_clean = run_single_model(
    model_name="RandomForest",
    prepared=prepared_clean,
    cfg=CFG,
    scenario="clean",
    severity=None,
)

print(rf_clean["metrics"])
save_result(rf_clean, "rf_clean.pkl")

In [ ]:
gwn_clean = run_single_model(
    model_name="GraphWaveNet",
    prepared=prepared_clean,
    cfg=CFG,
    scenario="clean",
    severity=None,
)

print(gwn_clean["metrics"])
save_result(gwn_clean, "gwn_clean.pkl")

In [ ]:
gwn_gru_clean = run_single_model(
    model_name="GraphWaveNet_GRU",
    prepared=prepared_clean,
    cfg=CFG,
    scenario="clean",
    severity=None,
)

print(gwn_gru_clean["metrics"])
save_result(gwn_gru_clean, "gwn_gru_clean.pkl")

In [ ]:
gwn_lstm_clean = run_single_model(
    model_name="GraphWaveNet_LSTM",
    prepared=prepared_clean,
    cfg=CFG,
    scenario="clean",
    severity=None,
)

print(gwn_lstm_clean["metrics"])
save_result(gwn_lstm_clean, "gwn_lstm_clean.pkl")

In [ ]:
gwn_gru_lstm_clean = run_single_model(
    model_name="GraphWaveNet_GRU_LSTM",
    prepared=prepared_clean,
    cfg=CFG,
    scenario="clean",
    severity=None,
)

print(gwn_gru_lstm_clean["metrics"])
save_result(gwn_gru_lstm_clean, "gwn_gru_lstm_clean.pkl")

In [ ]:
clean_results = {
    "RandomForest": rf_clean,
    "GraphWaveNet": gwn_clean,
    "GraphWaveNet_GRU": gwn_gru_clean,
    "GraphWaveNet_LSTM": gwn_lstm_clean,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_clean,
}

def single_result_metrics_table(results_dict, metric_name="MAE"):
    rows = {}
    for name, result in results_dict.items():
        rows[name] = result["metrics"][metric_name]
    df = pd.DataFrame(rows).T
    df = df[[h for h in CFG.horizons if h in df.columns]]
    return df

mae_clean_table = single_result_metrics_table(clean_results, metric_name="MAE")
rmse_clean_table = single_result_metrics_table(clean_results, metric_name="RMSE")

print("=== CLEAN MAE ===")
display(mae_clean_table)

print("=== CLEAN RMSE ===")
display(rmse_clean_table)

In [ ]:
base = mae_clean_table.loc["GraphWaveNet"]

improve_vs_gwn = pd.DataFrame({
    model: 100.0 * (base - mae_clean_table.loc[model]) / base
    for model in ["GraphWaveNet_GRU", "GraphWaveNet_LSTM", "GraphWaveNet_GRU_LSTM"]
}).T

print("=== RELATIVE MAE IMPROVEMENT VS GRAPHWAVENET (%) ===")
display(improve_vs_gwn)

## Sensor outage 30%: prepare experiment

In [35]:
prepared_outage_30 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="sensor_outage",
    severity=0.30,
)

In [ ]:
rf_outage_30 = run_single_model(
    model_name="RandomForest",
    prepared=prepared_outage_30,
    cfg=CFG,
    scenario="sensor_outage",
    severity=0.30,
)
save_result(rf_outage_30, "rf_outage_30.pkl")
print(rf_outage_30["metrics"])

In [ ]:
gwn_outage_30 = run_single_model(
    model_name="GraphWaveNet",
    prepared=prepared_outage_30,
    cfg=CFG,
    scenario="sensor_outage",
    severity=0.30,
)
save_result(gwn_outage_30, "gwn_outage_30.pkl")
print(gwn_outage_30["metrics"])

In [ ]:
gwn_gru_outage_30 = run_single_model(
    model_name="GraphWaveNet_GRU",
    prepared=prepared_outage_30,
    cfg=CFG,
    scenario="sensor_outage",
    severity=0.30,
)
save_result(gwn_gru_outage_30, "gwn_gru_outage_30.pkl")
print(gwn_gru_outage_30["metrics"])

In [ ]:
gwn_lstm_outage_30 = run_single_model(
    model_name="GraphWaveNet_LSTM",
    prepared=prepared_outage_30,
    cfg=CFG,
    scenario="sensor_outage",
    severity=0.30,
)
save_result(gwn_lstm_outage_30, "gwn_lstm_outage_30.pkl")
print(gwn_lstm_outage_30["metrics"])

In [ ]:
gwn_gru_lstm_outage_30 = run_single_model(
    model_name="GraphWaveNet_GRU_LSTM",
    prepared=prepared_outage_30,
    cfg=CFG,
    scenario="sensor_outage",
    severity=0.30,
)
save_result(gwn_gru_lstm_outage_30, "gwn_gru_lstm_outage_30.pkl")
print(gwn_gru_lstm_outage_30["metrics"])

In [ ]:
outage_30_results = {
    "RandomForest": rf_outage_30,
    "GraphWaveNet": gwn_outage_30,
    "GraphWaveNet_GRU": gwn_gru_outage_30,
    "GraphWaveNet_LSTM": gwn_lstm_outage_30,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_outage_30,
}

outage_30_mae = single_result_metrics_table(outage_30_results, metric_name="MAE")
print("=== SENSOR OUTAGE 30% | MAE ===")
display(outage_30_mae)

## Sensor outage 30% results

This section combines the individual model results for the 30% sensor outage scenario into one table for comparison.

In [ ]:
outage_30_results = {
    "RandomForest": rf_outage_30,
    "GraphWaveNet": gwn_outage_30,
    "GraphWaveNet_GRU": gwn_gru_outage_30,
    "GraphWaveNet_LSTM": gwn_lstm_outage_30,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_outage_30,
}

outage_30_mae = single_result_metrics_table(outage_30_results, metric_name="MAE")
outage_30_rmse = single_result_metrics_table(outage_30_results, metric_name="RMSE")

print("=== SENSOR OUTAGE 30% | MAE ===")
display(outage_30_mae)

print("=== SENSOR OUTAGE 30% | RMSE ===")
display(outage_30_rmse)

## Sensor outage 30% masked-only results

This table focuses only on the stations that were masked during the outage scenario.
This is especially important for understanding graph-based recovery behavior.

In [ ]:
def single_result_masked_table(results_dict, metric_name="MAE"):
    rows = {}
    for name, result in results_dict.items():
        if "masked_only_metrics" in result:
            rows[name] = result["masked_only_metrics"][metric_name]
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).T
    df = df[[h for h in CFG.horizons if h in df.columns]]
    return df

outage_30_masked_mae = single_result_masked_table(outage_30_results, metric_name="MAE")
outage_30_masked_rmse = single_result_masked_table(outage_30_results, metric_name="RMSE")

print("=== SENSOR OUTAGE 30% | MASKED-ONLY MAE ===")
display(outage_30_masked_mae)

print("=== SENSOR OUTAGE 30% | MASKED-ONLY RMSE ===")
display(outage_30_masked_rmse)

## Relative improvement over GraphWaveNet under sensor outage

Positive values indicate that the recurrent refinement model improves over the base GraphWaveNet under outage conditions.

In [ ]:
base_outage = outage_30_mae.loc["GraphWaveNet"]

improve_vs_gwn_outage_30 = pd.DataFrame({
    model: 100.0 * (base_outage - outage_30_mae.loc[model]) / base_outage
    for model in ["GraphWaveNet_GRU", "GraphWaveNet_LSTM", "GraphWaveNet_GRU_LSTM"]
}).T

print("=== RELATIVE MAE IMPROVEMENT VS GRAPHWAVENET | OUTAGE 30% ===")
display(improve_vs_gwn_outage_30)

## Sensor outage 10%: prepare experiment

In [ ]:
prepared_outage_10 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="sensor_outage",
    severity=0.10,
)

In [ ]:
rf_outage_10 = run_single_model("RandomForest", prepared_outage_10, CFG, scenario="sensor_outage", severity=0.10)
save_result(rf_outage_10, "rf_outage_10.pkl")

gwn_outage_10 = run_single_model("GraphWaveNet", prepared_outage_10, CFG, scenario="sensor_outage", severity=0.10)
save_result(gwn_outage_10, "gwn_outage_10.pkl")

gwn_gru_outage_10 = run_single_model("GraphWaveNet_GRU", prepared_outage_10, CFG, scenario="sensor_outage", severity=0.10)
save_result(gwn_gru_outage_10, "gwn_gru_outage_10.pkl")

gwn_lstm_outage_10 = run_single_model("GraphWaveNet_LSTM", prepared_outage_10, CFG, scenario="sensor_outage", severity=0.10)
save_result(gwn_lstm_outage_10, "gwn_lstm_outage_10.pkl")

gwn_gru_lstm_outage_10 = run_single_model("GraphWaveNet_GRU_LSTM", prepared_outage_10, CFG, scenario="sensor_outage", severity=0.10)
save_result(gwn_gru_lstm_outage_10, "gwn_gru_lstm_outage_10.pkl")

In [ ]:
outage_10_results = {
    "RandomForest": rf_outage_10,
    "GraphWaveNet": gwn_outage_10,
    "GraphWaveNet_GRU": gwn_gru_outage_10,
    "GraphWaveNet_LSTM": gwn_lstm_outage_10,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_outage_10,
}

outage_10_mae = single_result_metrics_table(outage_10_results, metric_name="MAE")
outage_10_masked_mae = single_result_masked_table(outage_10_results, metric_name="MAE")

print("=== SENSOR OUTAGE 10% | MAE ===")
display(outage_10_mae)

print("=== SENSOR OUTAGE 10% | MASKED-ONLY MAE ===")
display(outage_10_masked_mae)

## Sensor outage 20%: prepare experiment

In [ ]:
prepared_outage_20 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="sensor_outage",
    severity=0.20,
)

In [ ]:
rf_outage_20 = run_single_model("RandomForest", prepared_outage_20, CFG, scenario="sensor_outage", severity=0.20)
save_result(rf_outage_20, "rf_outage_20.pkl")

gwn_outage_20 = run_single_model("GraphWaveNet", prepared_outage_20, CFG, scenario="sensor_outage", severity=0.20)
save_result(gwn_outage_20, "gwn_outage_20.pkl")

gwn_gru_outage_20 = run_single_model("GraphWaveNet_GRU", prepared_outage_20, CFG, scenario="sensor_outage", severity=0.20)
save_result(gwn_gru_outage_20, "gwn_gru_outage_20.pkl")

gwn_lstm_outage_20 = run_single_model("GraphWaveNet_LSTM", prepared_outage_20, CFG, scenario="sensor_outage", severity=0.20)
save_result(gwn_lstm_outage_20, "gwn_lstm_outage_20.pkl")

gwn_gru_lstm_outage_20 = run_single_model("GraphWaveNet_GRU_LSTM", prepared_outage_20, CFG, scenario="sensor_outage", severity=0.20)
save_result(gwn_gru_lstm_outage_20, "gwn_gru_lstm_outage_20.pkl")

In [ ]:
outage_20_results = {
    "RandomForest": rf_outage_20,
    "GraphWaveNet": gwn_outage_20,
    "GraphWaveNet_GRU": gwn_gru_outage_20,
    "GraphWaveNet_LSTM": gwn_lstm_outage_20,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_outage_20,
}

outage_20_mae = single_result_metrics_table(outage_20_results, metric_name="MAE")
outage_20_masked_mae = single_result_masked_table(outage_20_results, metric_name="MAE")

print("=== SENSOR OUTAGE 20% | MAE ===")
display(outage_20_mae)

print("=== SENSOR OUTAGE 20% | MASKED-ONLY MAE ===")
display(outage_20_masked_mae)

## Sensor outage degradation plot

This plot compares how error changes as outage severity increases.

In [ ]:
sensor_plot_df = pd.DataFrame({
    "RandomForest": [
        outage_10_mae.loc["RandomForest", 72],
        outage_20_mae.loc["RandomForest", 72],
        outage_30_mae.loc["RandomForest", 72],
    ],
    "GraphWaveNet": [
        outage_10_mae.loc["GraphWaveNet", 72],
        outage_20_mae.loc["GraphWaveNet", 72],
        outage_30_mae.loc["GraphWaveNet", 72],
    ],
    "GraphWaveNet_GRU": [
        outage_10_mae.loc["GraphWaveNet_GRU", 72],
        outage_20_mae.loc["GraphWaveNet_GRU", 72],
        outage_30_mae.loc["GraphWaveNet_GRU", 72],
    ],
    "GraphWaveNet_LSTM": [
        outage_10_mae.loc["GraphWaveNet_LSTM", 72],
        outage_20_mae.loc["GraphWaveNet_LSTM", 72],
        outage_30_mae.loc["GraphWaveNet_LSTM", 72],
    ],
    "GraphWaveNet_GRU_LSTM": [
        outage_10_mae.loc["GraphWaveNet_GRU_LSTM", 72],
        outage_20_mae.loc["GraphWaveNet_GRU_LSTM", 72],
        outage_30_mae.loc["GraphWaveNet_GRU_LSTM", 72],
    ],
}, index=[0.10, 0.20, 0.30])

plt.figure(figsize=(8, 5))
for model in sensor_plot_df.columns:
    plt.plot(sensor_plot_df.index, sensor_plot_df[model], marker="o", label=model)

plt.xlabel("Sensor outage severity")
plt.ylabel("MAE at 72h")
plt.title("Sensor outage robustness at 72h")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Save outage tables

In [ ]:
outage_10_mae.to_csv("outage_10_mae.csv")
outage_20_mae.to_csv("outage_20_mae.csv")
outage_30_mae.to_csv("outage_30_mae.csv")

outage_10_masked_mae.to_csv("outage_10_masked_mae.csv")
outage_20_masked_mae.to_csv("outage_20_masked_mae.csv")
outage_30_masked_mae.to_csv("outage_30_masked_mae.csv")

improve_vs_gwn_outage_30.to_csv("improve_vs_gwn_outage_30.csv")

print("Saved outage tables.")

## Temporal missingness experiments

This scenario masks a fraction of time steps in the input window while keeping the station set fixed. It simulates telemetry gaps, communication delays, or partial time-series corruption.

In [ ]:
prepared_temporal_10 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="temporal_missingness",
    severity=0.10,
)

In [ ]:
rf_temporal_10 = run_single_model("RandomForest", prepared_temporal_10, CFG, scenario="temporal_missingness", severity=0.10)
save_result(rf_temporal_10, "rf_temporal_10.pkl")

gwn_temporal_10 = run_single_model("GraphWaveNet", prepared_temporal_10, CFG, scenario="temporal_missingness", severity=0.10)
save_result(gwn_temporal_10, "gwn_temporal_10.pkl")

gwn_gru_temporal_10 = run_single_model("GraphWaveNet_GRU", prepared_temporal_10, CFG, scenario="temporal_missingness", severity=0.10)
save_result(gwn_gru_temporal_10, "gwn_gru_temporal_10.pkl")

gwn_lstm_temporal_10 = run_single_model("GraphWaveNet_LSTM", prepared_temporal_10, CFG, scenario="temporal_missingness", severity=0.10)
save_result(gwn_lstm_temporal_10, "gwn_lstm_temporal_10.pkl")

gwn_gru_lstm_temporal_10 = run_single_model("GraphWaveNet_GRU_LSTM", prepared_temporal_10, CFG, scenario="temporal_missingness", severity=0.10)
save_result(gwn_gru_lstm_temporal_10, "gwn_gru_lstm_temporal_10.pkl")

In [ ]:
prepared_temporal_20 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="temporal_missingness",
    severity=0.20,
)

In [ ]:
rf_temporal_20 = run_single_model("RandomForest", prepared_temporal_20, CFG, scenario="temporal_missingness", severity=0.20)
save_result(rf_temporal_20, "rf_temporal_20.pkl")

gwn_temporal_20 = run_single_model("GraphWaveNet", prepared_temporal_20, CFG, scenario="temporal_missingness", severity=0.20)
save_result(gwn_temporal_20, "gwn_temporal_20.pkl")

gwn_gru_temporal_20 = run_single_model("GraphWaveNet_GRU", prepared_temporal_20, CFG, scenario="temporal_missingness", severity=0.20)
save_result(gwn_gru_temporal_20, "gwn_gru_temporal_20.pkl")

gwn_lstm_temporal_20 = run_single_model("GraphWaveNet_LSTM", prepared_temporal_20, CFG, scenario="temporal_missingness", severity=0.20)
save_result(gwn_lstm_temporal_20, "gwn_lstm_temporal_20.pkl")

gwn_gru_lstm_temporal_20 = run_single_model("GraphWaveNet_GRU_LSTM", prepared_temporal_20, CFG, scenario="temporal_missingness", severity=0.20)
save_result(gwn_gru_lstm_temporal_20, "gwn_gru_lstm_temporal_20.pkl")

In [ ]:
prepared_temporal_30 = prepare_experiment_objects(
    flow_df, speed_df, station_meta, CFG,
    scenario="temporal_missingness",
    severity=0.30,
)

In [ ]:
rf_temporal_30 = run_single_model("RandomForest", prepared_temporal_30, CFG, scenario="temporal_missingness", severity=0.30)
save_result(rf_temporal_30, "rf_temporal_30.pkl")

gwn_temporal_30 = run_single_model("GraphWaveNet", prepared_temporal_30, CFG, scenario="temporal_missingness", severity=0.30)
save_result(gwn_temporal_30, "gwn_temporal_30.pkl")

gwn_gru_temporal_30 = run_single_model("GraphWaveNet_GRU", prepared_temporal_30, CFG, scenario="temporal_missingness", severity=0.30)
save_result(gwn_gru_temporal_30, "gwn_gru_temporal_30.pkl")

gwn_lstm_temporal_30 = run_single_model("GraphWaveNet_LSTM", prepared_temporal_30, CFG, scenario="temporal_missingness", severity=0.30)
save_result(gwn_lstm_temporal_30, "gwn_lstm_temporal_30.pkl")

gwn_gru_lstm_temporal_30 = run_single_model("GraphWaveNet_GRU_LSTM", prepared_temporal_30, CFG, scenario="temporal_missingness", severity=0.30)
save_result(gwn_gru_lstm_temporal_30, "gwn_gru_lstm_temporal_30.pkl")

In [ ]:
temporal_10_results = {
    "RandomForest": rf_temporal_10,
    "GraphWaveNet": gwn_temporal_10,
    "GraphWaveNet_GRU": gwn_gru_temporal_10,
    "GraphWaveNet_LSTM": gwn_lstm_temporal_10,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_temporal_10,
}

temporal_20_results = {
    "RandomForest": rf_temporal_20,
    "GraphWaveNet": gwn_temporal_20,
    "GraphWaveNet_GRU": gwn_gru_temporal_20,
    "GraphWaveNet_LSTM": gwn_lstm_temporal_20,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_temporal_20,
}

temporal_30_results = {
    "RandomForest": rf_temporal_30,
    "GraphWaveNet": gwn_temporal_30,
    "GraphWaveNet_GRU": gwn_gru_temporal_30,
    "GraphWaveNet_LSTM": gwn_lstm_temporal_30,
    "GraphWaveNet_GRU_LSTM": gwn_gru_lstm_temporal_30,
}

temporal_10_mae = single_result_metrics_table(temporal_10_results, metric_name="MAE")
temporal_20_mae = single_result_metrics_table(temporal_20_results, metric_name="MAE")
temporal_30_mae = single_result_metrics_table(temporal_30_results, metric_name="MAE")

print("=== TEMPORAL MISSINGNESS 10% | MAE ===")
display(temporal_10_mae)

print("=== TEMPORAL MISSINGNESS 20% | MAE ===")
display(temporal_20_mae)

print("=== TEMPORAL MISSINGNESS 30% | MAE ===")
display(temporal_30_mae)